# Wan 2.2 TI2V-5B Batch Generation on Kaggle GPU (T4 / 2x T4)

This notebook runs the image-to-video moving shots stage for the **Heritage Short** pipeline.

### Workflow:
1. Upload `wan22_manifest.json` and the `keyframes/` directory (or zip) into the Kaggle working directory.
2. Toggle `DRAFT_MODE = True` for fast 480p preview or `False` for final 720p.
3. Run all cells: clips (`.mp4`), sidecars (`.meta.json`), and `kaggle_runs.jsonl` are saved after every clip.
4. Download `wan22_shots_bundle.zip` and extract into `outputs/shots/` locally.

In [ ]:
# Cell 1: Install dependencies
!pip install -q diffusers>=0.32.0 transformers>=4.48.0 accelerate>=0.34.0 torchvision imageio imageio-ffmpeg sentencepiece

In [ ]:
# Cell 2: Configuration & Toggles
DRAFT_MODE = True  # Set to False for final 720p

MANIFEST_PATH = "wan22_manifest.json"
KEYFRAMES_DIR = "keyframes"
OUTPUT_DIR = "outputs/shots"
KAGGLE_RUNS_LOG = f"{OUTPUT_DIR}/kaggle_runs.jsonl"
ZIP_OUTPUT = "wan22_shots_bundle.zip"

if DRAFT_MODE:
    RESOLUTION = (832, 480)
    NUM_INFERENCE_STEPS = 20
    GUIDANCE_SCALE = 5.0
else:
    RESOLUTION = (1280, 720)
    NUM_INFERENCE_STEPS = 35
    GUIDANCE_SCALE = 6.0

print(f"Configuration: DRAFT_MODE={DRAFT_MODE}, Resolution={RESOLUTION}, Steps={NUM_INFERENCE_STEPS}")

In [ ]:
# Cell 3: Load Pipeline with memory optimizations for T4 (16GB VRAM)
import torch
from diffusers import WanImageToVideoPipeline

print(f"CUDA available: {torch.cuda.is_available()}, Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

pipe = WanImageToVideoPipeline.from_pretrained(
    "Wan-AI/Wan2.2-TI2V-5B-Diffusers",
    torch_dtype=torch.float16,
)

try:
    pipe.enable_model_cpu_offload()
    print("[+] Model CPU offloading active.")
except Exception as e:
    print(f"[!] CPU offload fallback: {e}")
    pipe.to("cuda")

try:
    pipe.enable_vae_slicing()
    pipe.enable_vae_tiling()
    print("[+] VAE slicing & tiling active.")
except Exception:
    pass

In [ ]:
# Cell 4: Batch Video Generation Loop
import os
import json
import time
from datetime import datetime, timezone
from PIL import Image
from diffusers.utils import export_to_video

os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)

shots = manifest.get("shots", [])
negative_prompt = manifest.get("negative_prompt_video", "")
fps = manifest.get("fps", 24)

print(f"Starting generation for {len(shots)} shots...")

for idx, shot in enumerate(shots, 1):
    shot_id = shot["shot_id"]
    motion_prompt = shot["motion_prompt"]
    seed = int(shot.get("seed", 42000))
    target_duration_s = float(shot.get("target_duration_s", 5.0))
    input_hash = shot.get("input_hash", "")

    # Locate keyframe image
    candidates = [
        f"{KEYFRAMES_DIR}/{shot_id}.png",
        shot.get("keyframe_path", ""),
        f"{shot_id}.png",
    ]
    image_path = next((c for c in candidates if os.path.exists(c)), None)

    if not image_path:
        print(f"[{idx}/{len(shots)}] Keyframe missing for {shot_id}. Skipping.")
        continue

    output_mp4 = f"{OUTPUT_DIR}/{shot_id}.mp4"
    meta_json = f"{output_mp4}.meta.json"

    # Check cache
    if os.path.exists(output_mp4) and os.path.exists(meta_json):
        try:
            with open(meta_json, "r", encoding="utf-8") as mf:
                if json.load(mf).get("input_hash") == input_hash:
                    print(f"[{idx}/{len(shots)}] [CACHE HIT] {shot_id} already generated.")
                    continue
        except Exception:
            pass

    print(f"[{idx}/{len(shots)}] [GENERATING] {shot_id} | Seed: {seed} | Duration: {target_duration_s}s...")
    start_time = time.time()

    try:
        image = Image.open(image_path).convert("RGB")
        num_frames = max(17, int(round(target_duration_s * 16)))
        if num_frames % 4 != 1:
            num_frames = ((num_frames // 4) * 4) + 1

        generator = torch.Generator(device="cuda").manual_seed(seed)

        output = pipe(
            image=image,
            prompt=motion_prompt,
            negative_prompt=negative_prompt,
            height=RESOLUTION[1],
            width=RESOLUTION[0],
            num_frames=num_frames,
            num_inference_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            generator=generator,
        )

        frames = output.frames[0]
        export_to_video(frames, output_mp4, fps=fps)
        latency_s = time.time() - start_time
        gpu_min = latency_s / 60.0

        # Write immediate meta sidecar
        meta_payload = {
            "input_hash": input_hash,
            "shot_id": shot_id,
            "model": "Wan-AI/Wan2.2-TI2V-5B",
            "seed": seed,
            "motion_prompt": motion_prompt,
            "target_duration_s": target_duration_s,
            "resolution": f"{RESOLUTION[0]}x{RESOLUTION[1]}",
            "draft_mode": DRAFT_MODE,
            "latency_s": round(latency_s, 2),
            "gpu_min": round(gpu_min, 4),
        }
        with open(meta_json, "w", encoding="utf-8") as mf:
            json.dump(meta_payload, mf, indent=2)

        # Log to kaggle_runs.jsonl immediately
        log_record = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "stage": "shots",
            "shot_id": shot_id,
            "model": "Wan-AI/Wan2.2-TI2V-5B",
            "params": {
                "draft_mode": DRAFT_MODE,
                "resolution": f"{RESOLUTION[0]}x{RESOLUTION[1]}",
                "num_inference_steps": NUM_INFERENCE_STEPS,
                "guidance_scale": GUIDANCE_SCALE,
                "target_duration_s": target_duration_s,
            },
            "seed": seed,
            "latency_s": round(latency_s, 4),
            "status": "ok",
            "gpu_min": round(gpu_min, 4),
            "est_cost_usd": 0.0,
            "cache_hit": False,
            "error": None,
        }
        with open(KAGGLE_RUNS_LOG, "a", encoding="utf-8") as lf:
            lf.write(json.dumps(log_record, ensure_ascii=False) + "\n")

        print(f"  [+] Saved {output_mp4} ({latency_s:.1f}s, {gpu_min:.2f} GPU min)")

    except Exception as e:
        latency_s = time.time() - start_time
        gpu_min = latency_s / 60.0
        print(f"  [-] Error on {shot_id}: {e}")
        err_record = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "stage": "shots",
            "shot_id": shot_id,
            "model": "Wan-AI/Wan2.2-TI2V-5B",
            "params": {"draft_mode": DRAFT_MODE, "resolution": f"{RESOLUTION[0]}x{RESOLUTION[1]}"},
            "seed": seed,
            "latency_s": round(latency_s, 4),
            "status": "error",
            "gpu_min": round(gpu_min, 4),
            "est_cost_usd": 0.0,
            "cache_hit": False,
            "error": str(e),
        }
        with open(KAGGLE_RUNS_LOG, "a", encoding="utf-8") as lf:
            lf.write(json.dumps(err_record, ensure_ascii=False) + "\n")

In [ ]:
# Cell 5: Package Outputs for Download
import zipfile

print(f"Packaging outputs into '{ZIP_OUTPUT}'...")
with zipfile.ZipFile(ZIP_OUTPUT, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(OUTPUT_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, os.path.dirname(OUTPUT_DIR))
            zipf.write(file_path, arcname=arcname)

print(f"Done! Download '{ZIP_OUTPUT}' from the Kaggle file browser and unzip into outputs/shots/")